## Imports

In [11]:
from langchain_ollama import OllamaEmbeddings
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
import os
from dotenv import load_dotenv
import requests
from global_variables import LOCAL_MODEL_NAME, LOCAL_BASE_URL


## Globale Variablen

In [12]:
system_prompt = "Du bist ein Tutor für Vorlesungsinhalte. Beantworte Fragen nur auf Basis des bereitgestellten Kontexts. Erkläre klar, korrekt und verständlich. Wenn Informationen fehlen oder unsicher sind, sage das ausdrücklich. Erfinde nichts und spekuliere nicht. Nutze Fachbegriffe korrekt und erkläre sie kurz, wenn nötig. Gib falls Informationen aus der Fuktion search_lecture_docs entnommen werden das heißt dass die Informationen aus einer Datei kommen, immer den Dateipfad in folgendem Format an: *Quelle*: `quelle`. Beispiel: *Quelle*: `data/raw/somefile.txt`"

LOCAL = True

## Tools

Retreival Tool

In [13]:
# Das Tool stellt Anfragen an die VectorDB und bekommt die entsprechenden Chunks zurück
@tool('search_lecture_docs', description=""" Searches the lecture documents using semantic similarity and retrieves the most relevant text chunks for answering the user's question. Use this tool when the user asks a question that requires information from the lecture documents. The `query` parameter must contain a concise semantic search query representing the user's information need. Query rules: - Preserve important technical terms, concepts, names, and keywords. - Do not include instructions to the assistant. - Do not include unnecessary conversational text. - Do not include the user's entire conversation. - Formulate the query so that it is useful for semantic vector search. - If the user asks a conceptual question, keep the important concepts from the original question. Examples: User: "Was ist Retrieval Augmented Generation?" query: "Retrieval Augmented Generation" User: "Wie funktioniert der RecursiveCharacterTextSplitter?" query: "RecursiveCharacterTextSplitter Funktionsweise" User: "Was ist der Unterschied zwischen RAG und Fine-Tuning?" query: "Unterschied RAG Fine-Tuning" User: "Wie werden Embeddings für die Vektorsuche verwendet?" query: "Embeddings Vektorsuche Verwendung" If the user explicitly asks about a specific file, DO NOT use this tool to retrieve all chunks of that file. Use the dedicated file-retrieval tool instead. This tool performs semantic similarity search. It does not search metadata or retrieve all chunks belonging to a specific file. """)
def search_lecture_docs(query: str):
    response = requests.post(
    "http://127.0.0.1:8000/query",
    json={
        "query": query,
        "n": 3
    })
    return response.json()


@tool('get_file_info', description=""" Retrieves all information/chunks from a specific lecture document. IMPORTANT: The `filename` parameter MUST contain ONLY the filename explicitly mentioned by the user. Never pass the user's complete question or any additional text. If the user explicitly refers to a file, extract only the filename including its file extension and pass it as the `filename` parameter. Examples: User: "Was steht in der Datei informationen.txt?" filename: "informationen.txt" User: "Was steht in example.md?" filename: "example.md" User: "Erkläre mir den Inhalt von lecture_03.pdf." filename: "lecture_03.pdf" User: "Was wird in 01-introduction.md über RAG erklärt?" filename: "01-introduction.md" User: "Gib mir die Informationen aus /data/lectures/example.md." filename: "example.md" Do NOT include words such as: - "Was steht in" - "Datei" - "Erkläre" - "Inhalt von" - the user's complete question - the file path, if the user provides a path The filename must be passed exactly as extracted from the user's request, including its extension. This tool should ONLY be used when the user explicitly identifies a specific file. If no specific filename is mentioned, do not use this tool. """)
def get_file_info(filename: str):
    #print(f"get_file_info: {filename}")
    response = requests.post(
    "http://127.0.0.1:8000/document",
    json={
        "filename": filename
    })
    return response.json()

## Initialisierungen

In [14]:
# Model für Lokale Ollama Modelle
model_local = ChatOllama(
    base_url=LOCAL_BASE_URL,
    model=LOCAL_MODEL_NAME,
)
# Model für Nvidia NIM API
load_dotenv()
model_api = ChatOpenAI(
    model="meta/llama-3.1-8b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"],
)

agent = create_agent(model_local if LOCAL else model_api, tools=[search_lecture_docs, get_file_info], system_prompt=system_prompt)

In [15]:
prompt = str(input())
print('USER: ' + prompt + "\n")
result = agent.invoke({"messages": [("user", prompt)]})
print('AGENT: '+result["messages"][-1].content)

**USER**: Wie sind die Regelungen zu Urlaubstagen?

**AGENT**: Die Regelungen zu Urlaubstagen laut den vorliegenden Vorlesungsunterlagen lauten wie folgt:

* **Anspruch** – Mitarbeitende haben Anspruch auf **30 Urlaubstage pro Jahr**.  
* **Beantragung** – Urlaub muss **mindestens zwei Wochen im Voraus** beantragt werden.  
* **Resturlaub** – Nicht genommener Resturlaub soll **bis zum 31. März des Folgejahres** genommen werden.

*Quelle*: `data/processed/example.md`
